# Residential Units ETL — QA & Summary Analysis

Reads the output feature class and QA tables from `C:\GIS\ParcelHistory.gdb`  
to validate the ETL results and summarize unit counts by geography.

**Run order:** run the ETL (`main.py`) before opening this notebook.

In [ ]:
import sys
sys.path.insert(0, r'C:\Users\mbindl\Documents\GitHub\Reporting\residential_units_etl')

import arcpy
import pandas as pd
import numpy as np

from config import (
    OUTPUT_FC, CSV_YEARS, FC_APN, FC_YEAR, FC_UNITS, FC_COUNTY,
    GDB,
    QA_UNITS_BY_YEAR, QA_LOST_APNS, QA_APN_CROSSWALK,
    QA_SPATIAL_COMPLETENESS,
    SPATIAL_FIELDS,
)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:,.1f}'.format)
print('Setup complete')

---
## 1. Load Output Feature Class

In [ ]:
GEO_FIELDS = [
    'TOWN_CENTER', 'LOCATION_TO_TOWNCENTER',
    'TAZ', 'PLAN_ID', 'PLAN_NAME',
    'ZONING_ID', 'ZONING_DESCRIPTION', 'REGIONAL_LANDUSE',
    'WITHIN_TRPA_BNDY', 'WITHIN_BONUSUNIT_BNDY',
    'PARCEL_ACRES',
]

READ_FIELDS = [FC_APN, FC_YEAR, FC_UNITS, FC_COUNTY] + GEO_FIELDS

# Only read fields that actually exist
existing = {f.name for f in arcpy.ListFields(OUTPUT_FC)}
missing  = [f for f in READ_FIELDS if f not in existing]
if missing:
    print(f'WARNING — fields not in FC (skipped): {missing}')
read = [f for f in READ_FIELDS if f in existing]

yr_list = ', '.join(str(y) for y in CSV_YEARS)
rows = []
with arcpy.da.SearchCursor(OUTPUT_FC, read, f"{FC_YEAR} IN ({yr_list})") as cur:
    for row in cur:
        rows.append(dict(zip(read, row)))

df = pd.DataFrame(rows).rename(columns={
    FC_APN: 'APN', FC_YEAR: 'Year', FC_UNITS: 'Units', FC_COUNTY: 'County'
})
df['Units'] = df['Units'].fillna(0).astype(int)

print(f'Rows loaded : {len(df):,}')
print(f'Years       : {sorted(df["Year"].unique())}')
print(f'Total Units : {df["Units"].sum():,}')
df.head(3)

---
## 2. Units by Year — CSV vs FC

In [ ]:
# Load QA table (written by ETL Step 6)
qa_yr_rows = []
with arcpy.da.SearchCursor(QA_UNITS_BY_YEAR,
        ['Year','CSV_Total','FC_Total','Diff','Status']) as cur:
    for r in cur:
        qa_yr_rows.append(dict(zip(['Year','CSV_Total','FC_Total','Diff','Status'], r)))

df_yr = pd.DataFrame(qa_yr_rows).sort_values('Year').reset_index(drop=True)
df_yr['Match'] = df_yr['Diff'].apply(lambda d: '✓' if d == 0 else f'{d:+,}')

print(df_yr[['Year','CSV_Total','FC_Total','Diff','Status']].to_string(index=False))

---
## 3. Units by County & Year

In [ ]:
if 'County' in df.columns:
    piv = (df.groupby(['County','Year'])['Units']
             .sum()
             .unstack('Year')
             .fillna(0)
             .astype(int))
    piv['TOTAL'] = piv.sum(axis=1)
    print(piv.sort_values('TOTAL', ascending=False))
else:
    print('County field not loaded')

---
## 4. Units by Town Center & Year

TRPA boundary rows only (excludes out-of-basin parcels).

In [ ]:
if 'TOWN_CENTER' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['TC'] = df_trpa['TOWN_CENTER'].fillna('No Town Center')

    piv_tc = (df_trpa.groupby(['TC','Year'])['Units']
                .sum()
                .unstack('Year')
                .fillna(0)
                .astype(int))
    piv_tc['TOTAL'] = piv_tc.sum(axis=1)
    piv_tc = piv_tc.sort_values('TOTAL', ascending=False)
    print(piv_tc)
else:
    print('TOWN_CENTER or WITHIN_TRPA_BNDY not available')

---
## 5. Units by Location-to-Town-Center & Year

In [ ]:
if 'LOCATION_TO_TOWNCENTER' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['Loc'] = df_trpa['LOCATION_TO_TOWNCENTER'].fillna('Unknown')

    piv_loc = (df_trpa.groupby(['Loc','Year'])['Units']
                 .sum()
                 .unstack('Year')
                 .fillna(0)
                 .astype(int))
    piv_loc['TOTAL'] = piv_loc.sum(axis=1)
    piv_loc = piv_loc.sort_values('TOTAL', ascending=False)
    print(piv_loc)
else:
    print('LOCATION_TO_TOWNCENTER or WITHIN_TRPA_BNDY not available')

---
## 6. Units by Regional Land Use

In [ ]:
if 'REGIONAL_LANDUSE' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['RLU'] = df_trpa['REGIONAL_LANDUSE'].fillna('Unknown')

    piv_rlu = (df_trpa.groupby(['RLU','Year'])['Units']
                 .sum()
                 .unstack('Year')
                 .fillna(0)
                 .astype(int))
    piv_rlu['TOTAL'] = piv_rlu.sum(axis=1)
    piv_rlu = piv_rlu.sort_values('TOTAL', ascending=False)
    print(piv_rlu)
else:
    print('REGIONAL_LANDUSE or WITHIN_TRPA_BNDY not available')

---
## 7. Units by TAZ

In [ ]:
if 'TAZ' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()

    taz_yr = (df_trpa[df_trpa['TAZ'].notna()]
                .groupby(['TAZ','Year'])['Units']
                .sum()
                .unstack('Year')
                .fillna(0)
                .astype(int))
    taz_yr['TOTAL'] = taz_yr.sum(axis=1)
    taz_yr = taz_yr.sort_values('TOTAL', ascending=False)

    print(f'TAZ count: {len(taz_yr)}')
    print(taz_yr.head(20))
else:
    print('TAZ or WITHIN_TRPA_BNDY not available')

---
## 8. Units by Zoning

In [ ]:
if 'ZONING_DESCRIPTION' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['Zone'] = df_trpa['ZONING_DESCRIPTION'].fillna('Unknown')

    zone_sum = (df_trpa.groupby('Zone')['Units']
                  .sum()
                  .reset_index()
                  .sort_values('Units', ascending=False))
    print(zone_sum.head(20).to_string(index=False))
else:
    print('ZONING_DESCRIPTION or WITHIN_TRPA_BNDY not available')

---
## 9. Spatial Completeness (TRPA boundary rows only)

In [ ]:
if arcpy.Exists(QA_SPATIAL_COMPLETENESS):
    sp_rows = []
    flds = ['Field','Null_Count','Pct_Null','TRPA_Rows','Status']
    with arcpy.da.SearchCursor(QA_SPATIAL_COMPLETENESS, flds) as cur:
        for r in cur:
            sp_rows.append(dict(zip(flds, r)))
    df_sp = pd.DataFrame(sp_rows).sort_values('Pct_Null', ascending=False)
    print(df_sp.to_string(index=False))
else:
    print('QA_Spatial_Completeness table not found — run ETL Step 6 first')

---
## 10. Lost APNs Summary

In [ ]:
if arcpy.Exists(QA_LOST_APNS):
    la_flds = ['APN','COUNTY','Years_Lost','Num_Years_Lost',
               'Total_Units_CSV','Years_In_FC','Issue_Category','Suggested_Action']
    la_rows = []
    with arcpy.da.SearchCursor(QA_LOST_APNS, la_flds) as cur:
        for r in cur:
            la_rows.append(dict(zip(la_flds, r)))
    df_lost = pd.DataFrame(la_rows)

    print(f'Total lost APNs: {len(df_lost):,}')
    print(f'Total lost units: {df_lost["Total_Units_CSV"].sum():,}')
    print()
    cat_sum = (df_lost.groupby('Issue_Category')
                 .agg(APNs=('APN','count'), Units=('Total_Units_CSV','sum'))
                 .sort_values('Units', ascending=False))
    print(cat_sum)
else:
    print('QA_Lost_APNs table not found — run ETL first')

In [ ]:
# Top lost APNs by units
if 'df_lost' in dir() and len(df_lost):
    print('Top 20 lost APNs by units:')
    top = df_lost.nlargest(20, 'Total_Units_CSV')[[
        'APN','COUNTY','Issue_Category','Total_Units_CSV','Years_Lost','Suggested_Action'
    ]]
    pd.set_option('display.max_colwidth', 80)
    print(top.to_string(index=False))

---
## 11. Bonus Unit Area Summary

In [ ]:
if 'WITHIN_BONUSUNIT_BNDY' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1]

    bonus_yr = (df_trpa.groupby(['WITHIN_BONUSUNIT_BNDY','Year'])['Units']
                  .sum()
                  .unstack('Year')
                  .fillna(0)
                  .astype(int))
    bonus_yr.index = bonus_yr.index.map({0: 'Outside Bonus Area', 1: 'Within Bonus Area'})
    bonus_yr['TOTAL'] = bonus_yr.sum(axis=1)
    print(bonus_yr)
else:
    print('WITHIN_BONUSUNIT_BNDY not available')

---
## 12. Year-over-Year Change by Town Center

In [ ]:
if 'TOWN_CENTER' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['TC'] = df_trpa['TOWN_CENTER'].fillna('No Town Center')

    piv = (df_trpa.groupby(['TC','Year'])['Units']
             .sum()
             .unstack('Year')
             .fillna(0)
             .astype(int))

    years = sorted(CSV_YEARS)
    chg = pd.DataFrame(index=piv.index)
    for i in range(1, len(years)):
        y0, y1 = years[i-1], years[i]
        if y0 in piv.columns and y1 in piv.columns:
            chg[f'{y0}→{y1}'] = piv[y1] - piv[y0]

    print('Year-over-year unit change by Town Center:')
    print(chg.sort_values(chg.columns[-1], ascending=False))
else:
    print('TOWN_CENTER or WITHIN_TRPA_BNDY not available')

---
## 13. Parcel Count vs Unit Count by Year (density check)

In [ ]:
if 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1]

    density = (df_trpa.groupby('Year')
                 .agg(
                     Parcels=('APN','count'),
                     Parcels_w_Units=('Units', lambda x: (x > 0).sum()),
                     Total_Units=('Units','sum'),
                 )
                 .reset_index())
    density['Pct_w_Units'] = (density['Parcels_w_Units'] / density['Parcels'] * 100).round(1)
    density['Avg_Units']   = (density['Total_Units'] / density['Parcels_w_Units']).round(2)
    print(density.to_string(index=False))